## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-08-19-50-07 +0000,nyt,"Trains Collide Near Cairo, Killing 4",https://www.nytimes.com/2026/11/08/world/middl...
1,2026-11-08-19-45-00 +0000,wapo,"In rare GOP rebuke, congressman says Trump mus...",https://www.washingtonpost.com/politics/2026/1...
2,2026-11-08-19-44-26 +0000,nyt,What Talarico’s Win Reveals About Christianity...,https://www.nytimes.com/2026/11/08/us/james-ta...
3,2026-11-08-19-41-25 +0000,cbc,Americans in Canada hope midterm results will ...,https://www.cbc.ca/news/canada/toronto/america...
4,2026-11-08-19-36-41 +0000,bbc,Top House Democrat 'a hard yes' on impeaching ...,https://www.bbc.co.uk/news/articles/cj4g1nwx39...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2479/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
9,trump,28
45,new,17
200,white,14
32,house,13
57,democrats,10


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
181,2026-11-08-00-16-00 +0000,wsj,UFC CEO Dana White said plans to meet with Pre...,https://www.wsj.com/politics/policy/ufc-chief-...,124
74,2026-11-08-15-21-05 +0000,nypost,House leader Hakeem Jeffries says Dems will re...,https://nypost.com/2026/11/08/us-news/hakeem-j...,111
183,2026-11-08-00-08-30 +0000,wapo,Trump says he offered White House press secret...,https://www.washingtonpost.com/politics/2026/1...,101
68,2026-11-08-15-57-44 +0000,chitri,What Donald Trump’s new White House chief of s...,https://www.chicagotribune.com/2026/11/08/step...,92
52,2026-11-08-17-00-00 +0000,wsj,"Justice Department lawyer Stanley Woodward, a ...",https://www.wsj.com/us-news/law/how-a-former-d...,86


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
181,124,2026-11-08-00-16-00 +0000,wsj,UFC CEO Dana White said plans to meet with Pre...,https://www.wsj.com/politics/policy/ufc-chief-...
74,70,2026-11-08-15-21-05 +0000,nypost,House leader Hakeem Jeffries says Dems will re...,https://nypost.com/2026/11/08/us-news/hakeem-j...
52,49,2026-11-08-17-00-00 +0000,wsj,"Justice Department lawyer Stanley Woodward, a ...",https://www.wsj.com/us-news/law/how-a-former-d...
209,43,2026-11-07-21-01-09 +0000,nypost,Shocking footage shows moment Krakow police sh...,https://nypost.com/2026/11/07/world-news/shock...
163,43,2026-11-08-02-00-00 +0000,wsj,A dominant midterm performance has given Democ...,https://www.wsj.com/politics/elections/the-new...
208,32,2026-11-07-21-04-58 +0000,nypost,Gov. Gavin Newsom reveals the moment he decide...,https://nypost.com/2026/11/07/us-news/gov-gavi...
176,32,2026-11-08-00-43-48 +0000,nypost,Lunatic who killed 6 in Poland mass shooting h...,https://nypost.com/2026/11/07/world-news/very-...
197,30,2026-11-07-22-18-09 +0000,wapo,"Ex-military chief Gadi Eisenkot, opponent of N...",https://www.washingtonpost.com/world/2026/11/0...
161,28,2026-11-08-02-46-33 +0000,nypost,Elizabeth Warren hates Wall Street — expect he...,https://nypost.com/2026/11/07/business/elizabe...
173,27,2026-11-08-00-54-22 +0000,chitri,"Strong earthquake shakes central Japan, shatte...",https://www.chicagotribune.com/2026/11/07/cent...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
